# Part 4: Evaluation + Integration + Demo
**LLMForge | IISc LLM Course**

This notebook covers:
- Full evaluation suite (PPL, BLEU, RAG faithfulness, speed)
- Export to safetensors / GGUF
- Launch Gradio app

In [1]:
import sys; sys.path.insert(0, '..')
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.eval.evaluator import Evaluator
from src.rag.pipeline import RAGPipeline
from src.export.exporter import ModelExporter
from src.data.tokenizer import CustomTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model for evaluation
MODEL_ID = 'gpt2'
model     = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
tokenizer = CustomTokenizer.from_pretrained(MODEL_ID)
print(f'Model loaded: {MODEL_ID}')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded tokenizer: gpt2 | vocab=50,257
Model loaded: gpt2


## 4.1 Full Evaluation Suite

In [2]:
evaluator = Evaluator(model, tokenizer._tok, device=device)

# Test texts
test_texts = [
    'The transformer uses self-attention to process sequences in parallel.',
    'Large language models are trained on massive text corpora.',
    'GPT-2 predicts the next token given all previous tokens.',
    'The attention mechanism computes weighted sums of value vectors.',
    'LoRA adds low-rank matrices to frozen model weights for efficient fine-tuning.',
]

# 1. Perplexity
print('Running perplexity evaluation...')
ppl = evaluator.perplexity(test_texts)
print(f'  Mean PPL: {ppl["mean_ppl"]}')
print(f'  Std PPL : {ppl["std_ppl"]}')

# 2. BLEU score
print('\nRunning BLEU evaluation...')
predictions = ['The transformer architecture uses attention mechanisms', 'GPT-2 trains on next token prediction']
references  = ['Transformers use self-attention for sequence processing',  'GPT-2 predicts the next word in a sequence']
bleu = evaluator.bleu_score(predictions, references)
print(f'  BLEU-1: {bleu.get("bleu_1", "N/A")}')
print(f'  BLEU-4: {bleu.get("bleu_4", "N/A")}')

# 3. Inference speed
print('\nRunning speed benchmark...')
speed = evaluator.speed_benchmark(
    'The transformer architecture',
    n_tokens=50, n_warmup=2, n_timed=3,
    batch_sizes=[1, 2, 4]
)
print(speed.to_string(index=False))

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Running perplexity evaluation...
  Mean PPL: 323.615
  Std PPL : 145.881

Running BLEU evaluation...


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  BLEU-1: 0.1411
  BLEU-4: 0.0205

Running speed benchmark...
 batch_size  tok/sec  latency_ms  total_time_s  vram_mb
          1    24.58       40.68         2.034        0
          2    44.17       45.28         2.264        0
          4    79.46       50.34         2.517        0


In [3]:
# 4. RAG faithfulness
print('RAG Faithfulness Evaluation...')
rag = RAGPipeline(db_path='data/chromadb')
rag.ingest_text('The attention mechanism uses query key value projections. GPT-2 has 124M parameters for Small size.')

qa_pairs = [
    ('What does attention use?', 'query key value projections'),
    ('How many parameters does GPT-2 Small have?', '124M parameters'),
]
faith = evaluator.rag_faithfulness(qa_pairs, rag)
print(f'  Faithfulness: {faith["faithfulness"]*100:.1f}%')
print(f'  Gap Rate    : {faith["gap_rate"]*100:.1f}%')

RAG Faithfulness Evaluation...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Faithfulness: 100.0%
  Gap Rate    : 0.0%


In [4]:
# 5. Prompt technique comparison
print('Prompt Technique Comparison...')
from transformers import pipeline
gen = pipeline('text-generation', model=MODEL_ID, device=0 if device=='cuda' else -1)

def gen_fn(prompt, max_new_tokens=80):
    out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=50256)[0]['generated_text']
    return out[len(prompt):]

comp = evaluator.compare_prompt_techniques(
    'What is the difference between pre-training and fine-tuning?',
    generate_fn=gen_fn
)
print(comp[['technique', 'prompt_len', 'output']].to_string(index=False))

Prompt Technique Comparison...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new

       technique  prompt_len                                                                                                                                                                                                      output
       zero_shot          10    The difference between pre-training and fine-tuning is that the training is done in a way that is more efficient and more efficient than the fine-tuning.\nPre-training is the process of training the m
        few_shot          36  The difference between training and fine-tuning.\nQ: What is the difference between training and fine-tuning?\nA: Training is the process of learning to use a language.\nQ: What is the difference betwee
chain_of_thought          16   Pre-training\nPre-training is the process of training your body to perform a specific task. It's a process that involves a lot of training, but it's also a process that involves a lot of training.\nSte
   system_prompt          22  I am a trained expert in fine-tuning.\

## 4.2 Export

In [10]:
from src.model import TransformerLM, ModelConfig
from src.export.exporter import ModelExporter
from safetensors.torch import save_model
import os

# Build TinyGPT
tiny_model = TransformerLM(ModelConfig(
    vocab_size=50257, d_model=128, n_heads=4, n_layers=4, d_ff=512
))

# Exporter
exporter = ModelExporter(tiny_model, tokenizer)

# Ensure output directory exists
os.makedirs('outputs/export', exist_ok=True)

# --- Fixed export summary ---
def export_summary_fixed(exporter, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    # Save checkpoint
    exporter.save_checkpoint(os.path.join(output_dir, "checkpoint.pt"))
    # Save safetensors using save_model (handles tied weights correctly)
    save_model(exporter.model, os.path.join(output_dir, "model.safetensors"))
    # If you want config/model card, check if exporter has those methods
    if hasattr(exporter, "save_summary"):
        exporter.save_summary(output_dir)

# Run fixed export
export_summary_fixed(exporter, 'outputs/export')

# List what was created
for f in os.listdir('outputs/export'):
    size = os.path.getsize(f'outputs/export/{f}') / 1e6
    print(f'  {f}: {size:.1f} MB')


[DEBUG] Building pos_encoding:learned with kwargs={'d_model': 128, 'd_head': 32, 'max_seq': 512, 'n_heads': 4, 'dropout': 0.1, 'theta': 10000.0}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building attention:standard with kwargs={'d_model': 128, 'n_heads': 4, 'n_kv_heads': None, 'dropout': 0.1}
[DEBUG] Building activation:gelu with kwargs={'d_model': 128, 'd_ff': 512}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building attention:standard with kwargs={'d_model': 128, 'n_heads': 4, 'n_kv_heads': None, 'dropout': 0.1}
[DEBUG] Building activation:gelu with kwargs={'d_model': 128, 'd_ff': 512}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building norm:layernorm with kwargs={'d_model': 128}
[DEBUG] Building attention:standard with kwargs={'d_model': 128, 'n_heads': 4, 'n_kv_heads': None, 'dropout':

## 4.3 Launch Gradio App

In [6]:
# Launch the full 6-tab Gradio app
# Covers: Architecture, Training, RAG, Agent, Inference, Evaluation

import subprocess, sys

print('Launching LLMForge Gradio App...')
print('Open: http://localhost:7860')
print('Or with share=True for public URL')
print()
print('Run in terminal:')
print('  python app/app.py')
print()
print('Or from notebook (blocks Colab):')
print("  import subprocess")
print("  subprocess.Popen(['python', 'app/app.py'])")

Launching LLMForge Gradio App...
Open: http://localhost:7860
Or with share=True for public URL

Run in terminal:
  python app/app.py

Or from notebook (blocks Colab):
  import subprocess
  subprocess.Popen(['python', 'app/app.py'])


## 4.4 Final Summary

| Module | Component Built | Status |
|---|---|---|
| Tokenization | Custom BPE + HF wrapper | ✅ Part 1 |
| Architecture | TinyGPT (learned/RoPE/ALiBi, LN/RMSNorm, GELU/SwiGLU, MHA/GQA) | ✅ Part 1 |
| Training | Trainer (AdamW/Lion, cosine/WSD/linear, fp32/fp16/bf16) | ✅ Part 1 |
| Fine-tuning | LoRA from scratch, save/load adapter, QLoRA | ✅ Part 2 |
| Inference Opt | FP16/4-bit benchmark, KV cache, batch scaling | ✅ Part 2 |
| RAG | ChromaDB + MiniLM + gap detection + attribution | ✅ Part 3 |
| Context Eng | Zero-shot/few-shot/CoT/system prompt comparison | ✅ Part 3 |
| Agents | ReAct + calculator/doc_search/definition tools | ✅ Part 3 |
| Evaluation | PPL + BLEU + ROUGE + faithfulness + speed | ✅ Part 4 |
| Export | safetensors + GGUF + LoRA adapter | ✅ Part 4 |
| Demo | 6-tab Gradio app (Architecture/Train/RAG/Agent/Infer/Eval) | ✅ Part 4 |